### Men's Betting Odds

Scrape implied odds of each team winning national championship pre tournament

In [1]:
SEASON = 2025
INCLUDE_CURRENT_SEASON = False

In [2]:
import pandas as pd
from tqdm.autonotebook import trange

pd.set_option('display.max_columns', 100)

def get_betting_odds(season: int) -> pd.DataFrame:
    d = pd.read_html(f'https://www.sportsoddshistory.com/cbb-main/?y={season-1}-{season}&sa=cbb&a=nc&o=r')[0]

    d.columns = [i[1] for i in d.columns]

    d.insert(0, 'Season', season)

    return d[['Season', 'Team', 'Round 1']]

df = pd.concat(
    [
       get_betting_odds(season)
       for season in trange(2012, SEASON + INCLUDE_CURRENT_SEASON)
       if season != 2020  # cancelled
    ],
    ignore_index=True
)

df

C:\Users\mhugh\AppData\Local\Temp\ipykernel_24596\3765012408.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange


  0%|          | 0/13 [00:00<?, ?it/s]

,Season,Team,Round 1
0,2012,Kentucky,185.0
1,2012,Kansas,1000.0
2,2012,Ohio State,650.0
3,2012,Louisville,3300.0
4,2012,Syracuse,1200.0
...,...,...,...
2412,2024,Louisville,NaN
2413,2024,Air Force,NaN
2414,2024,DePaul,NaN
2415,2024,Fresno State,NaN


Remove FIELD

In [4]:
df = df.loc[df['Team'] != 'FIELD', :].reset_index(drop=True)

df

,Season,Team,Round 1
0,2012,Kentucky,185.0
1,2012,Kansas,1000.0
2,2012,Ohio State,650.0
3,2012,Louisville,3300.0
4,2012,Syracuse,1200.0
...,...,...,...
2409,2024,Louisville,NaN
2410,2024,Air Force,NaN
2411,2024,DePaul,NaN
2412,2024,Fresno State,NaN


In [5]:
def american_odds_to_probability(odds):
    """
    Converts American betting odds to implied probability.
    
    Args:
        odds (int or float): The American odds value (e.g., -150, +200).

    Returns:
        float: The implied probability as a decimal (e.g., 0.60 for 60%).
    """
    if odds > 0:
        # Positive odds: (100 / (odds + 100))
        probability = 100 / (odds + 100)
    else:
        # Negative odds: (-odds / (-odds + 100))
        probability = -odds / (-odds + 100)
    
    return probability

df['Implied Champion Probability'] = df['Round 1'].apply(american_odds_to_probability)

df

,Season,Team,Round 1,Implied Champion Probability
0,2012,Kentucky,185.0,0.350877
1,2012,Kansas,1000.0,0.090909
2,2012,Ohio State,650.0,0.133333
3,2012,Louisville,3300.0,0.029412
4,2012,Syracuse,1200.0,0.076923
...,...,...,...,...
2409,2024,Louisville,NaN,NaN
2410,2024,Air Force,NaN,NaN
2411,2024,DePaul,NaN,NaN
2412,2024,Fresno State,NaN,NaN


Remove empty rows

In [6]:
df = df.loc[
    df['Implied Champion Probability'].notna(), 
    ['Season', 'Team', 'Implied Champion Probability']
].reset_index(drop=True)

df

,Season,Team,Implied Champion Probability
0,2012,Kentucky,0.350877
1,2012,Kansas,0.090909
2,2012,Ohio State,0.133333
3,2012,Louisville,0.029412
4,2012,Syracuse,0.076923
...,...,...,...
750,2024,Montana State,0.000500
751,2024,South Dakota State,0.000500
752,2024,St Peter's,0.000500
753,2024,Stetson,0.000500


In [7]:
df.to_parquet('../data/preprocessed/mens_betting/betting.parquet')

'Done'

'Done'